# Imports

In [1]:
from cyvcf2 import VCF
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD

# Calculate PCs

In [2]:
vcf = VCF("../data/ps3_gwas.vcf.gz")

samples = vcf.samples
n_samples = len(samples)

# Collect SNPs into list
genotype_list = []
for variant in vcf:
    g = np.array([gt[0] + gt[1] if gt[0] >= 0 and gt[1] >= 0 else np.nan
                  for gt in variant.genotypes])
    genotype_list.append(g)

# Convert to matrix, shape = (n_samples,n_snps)
X = np.vstack(genotype_list).T

# Data preprocessing
valid = ~np.isnan(X).all(axis=0)
X = X[:, valid]

X_imputed = np.where(np.isnan(X), np.nanmean(X, axis=0), X)

std = X_imputed.std(axis=0, ddof=1)
valid = (std > 0) & (~np.isnan(std))
X_imputed = X_imputed[:, valid]
std = std[valid]

X_centered = X_imputed - X_imputed.mean(axis=0)
X_std = X_centered / std

# PCA via SVD for first 3 PCs
k = 3
svd = TruncatedSVD(n_components=k)
U_times_S = svd.fit_transform(X_std)
S = svd.singular_values_

PCs = U_times_S

# Enforce consistent sign
PCs_fixed = PCs.copy()
for i in range(k):
    if PCs_fixed[:, i].sum() < 0:
        PCs_fixed[:, i] *= -1

# Print eigenvalues
p = X_std.shape[1]
eigenvals = (S**2) / p
print("Eigenvalues:", eigenvals)

# Write eigenvectors file
pcs_df = pd.DataFrame(PCs_fixed, columns=[f"PC{i+1}" for i in range(k)])
pcs_df.insert(0, "IID", samples)
pcs_df.insert(0, "FID", samples)
pcs_df.to_csv("../python_results/eigenvec.txt", sep=" ", header=False, index=False)

Eigenvalues: [26.46134596  0.92377561  0.91535254]


# Check PCs compared to PLINK

In [3]:
# Check correlation
plink = np.loadtxt("../plink_results/ps3_gwas.eigenvec", usecols=[2,3,4])
ours = PCs

corr = np.corrcoef(plink[:,0], ours[:,0])[0,1]
print("Correlation:", corr)

Correlation: -0.9999997647289756
